# Notebook 06 — Evaluation & Interpretation
Load the best disaster-level model from notebook 04.
Produce: confusion matrix, classification report, feature importances, SHAP values, and critical reflection.

**Business question:** At the moment of federal declaration, which funding tier will this disaster ultimately reach?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import sys
sys.path.append('../../')
from utils import classification_metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
sns.set_theme(style='whitegrid')

PROCESSED = '../../data/processed/'

with open(PROCESSED + 'best_disaster_model.pkl', 'rb') as f:
    d_bundle = pickle.load(f)

print('Disaster model :', d_bundle['model_name'])
print('Val  F1_weighted:', d_bundle['val_f1'])
print('Test F1_weighted:', d_bundle['test_f1'])

## 6.1 Confusion Matrix
Rows = actual tier, Columns = predicted tier.
Adjacent-tier errors (e.g. Moderate predicted as Minor) are acceptable;
extreme jumps (Minor predicted as Catastrophic) are not.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cm   = confusion_matrix(d_bundle['y_test'], d_bundle['preds'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=d_bundle['target_names'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'{d_bundle["model_name"]} — Disaster-Level (Test Set 2018+)', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=9)
plt.tight_layout()
plt.savefig(PROCESSED + 'confusion_matrix_disaster.png', dpi=150)
plt.show()

## 6.2 Classification Report — Per-Class Precision, Recall, F1
Pay particular attention to **recall on Tier 2 (Major) and Tier 3 (Catastrophic)** —
these are the highest-stakes misclassifications for budget planning.

In [ ]:
print(f"\n{'='*55}")
print(f"  {d_bundle['model_name']} — Disaster-Level (Test Set)")
print(f"{'='*55}")
print(classification_report(d_bundle['y_test'], d_bundle['preds'],
                            target_names=d_bundle['target_names'],
                            zero_division=0))

## 6.3 Val vs Test Performance — Overfitting Check
Compares validation F1 (used for model selection) with test F1 (final reporting).
A large gap would indicate overfitting to the validation period.

In [ ]:
summary = pd.DataFrame([{
    'Model':   d_bundle['model_name'],
    'Level':   'Disaster',
    'Val F1':  round(d_bundle['val_f1'],  4),
    'Test F1': round(d_bundle['test_f1'], 4),
    'Gap':     round(d_bundle['val_f1'] - d_bundle['test_f1'], 4),
}])
display(summary)

m = classification_metrics(d_bundle['y_test'].values, d_bundle['preds'],
                            label=d_bundle['model_name'],
                            target_names=d_bundle['target_names'])
print(f"\nFinal Test — Accuracy: {m['Accuracy']:.4f}  |  F1_weighted: {m['F1_weighted']:.4f}")

## 6.4 SHAP Values — Why Does the Model Predict What It Predicts?
SHAP explains individual predictions — which features push a value up or down.
Install with: `pip install shap`


In [ ]:
try:
    import shap

    pipeline = d_bundle['pipeline']
    X_test   = d_bundle['X_test']

    X_transformed = pipeline.named_steps['pre'].transform(X_test)
    pre           = pipeline.named_steps['pre']
    ohe_names     = pre.transformers_[0][1].named_steps['ohe'].get_feature_names_out(d_bundle['cat_features'])
    all_names     = list(ohe_names) + d_bundle['num_features']

    best_model = pipeline.named_steps['model']
    explainer  = shap.TreeExplainer(best_model)

    idx = np.random.choice(len(X_transformed), size=min(300, len(X_transformed)), replace=False)
    shap_values = explainer.shap_values(X_transformed[idx])

    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # Class 1 = Moderate (most populated tier)

    shap.summary_plot(shap_values, X_transformed[idx],
                      feature_names=all_names, max_display=15, show=True)

except ImportError:
    print('shap not installed. Run: pip install shap')

## 6.5 Critical Reflection

### Business Problem
When a federal disaster is declared, FEMA and state emergency managers must immediately
make staffing, contracting, and budget reservation decisions — but the total federal PA
cost won't be known for 12–18 months while project applications are still filed. This model
predicts the disaster's ultimate funding tier using only information available at declaration time.

### What the Model Does Well
- **+30% F1 over stratified baseline** on held-out post-2018 data — the model learns
  genuine signal, not historical memorisation.
- **Moderate tier recall = 88%**: the most common class is predicted reliably, meaning
  routine budget sizing decisions will be correct the majority of the time.
- **Prospective validity**: all features (`n_counties`, `declaration_lag_days`,
  `incidentType`, demographics, risk score) are available at or shortly after declaration.
- **CPI-adjusted labels**: tier thresholds are inflation-adjusted to 2019 dollars,
  ensuring consistent classification across the 25-year dataset.
- **Proper model selection**: best model chosen on validation set (2016–2017);
  test set (2018+) touched only once for final reporting.

### Class Imbalance
- Moderate disasters (Tier 1) dominate (~60% of events); Catastrophic (Tier 3) are rare (~8%).
- `class_weight='balanced'` was applied to prevent the model ignoring minority tiers.
- Despite this, **Major (Tier 2) and Catastrophic (Tier 3) recall remains near 0%** —
  the model almost never correctly flags the highest-cost events ahead of time.

### Key Limitation — High-Tier Recall
Missing a Catastrophic event is the costliest error: a Catastrophic disaster misclassified
as Moderate represents a >$450M under-reservation of federal funds. This reflects the
absence of physical intensity features (wind speed, precipitation, earthquake magnitude)
that most directly predict disaster scale — not a methodological flaw.

### Construct Validity
Tier thresholds are anchored to 2019 policy values. Only the Small/Large project boundary
($131,100) is subject to inflationary drift; the disaster-level thresholds ($1M / $50M / $500M)
are stable. CPI adjustment of dollar amounts before binning addresses the core of this concern.

### Limitations
- **No physical intensity signal**: NOAA Storm Events data (wind speed, precipitation,
  storm track) would be the primary improvement.
- **n_counties may grow post-declaration**: affected county count can increase in the days
  following declaration, introducing minor look-ahead noise.
- **Static demographics**: Census and NRI features are single-snapshot values.

### Future Work
- **NOAA Storm Events integration** for physical intensity features.
- **Cost-sensitive learning**: penalise Catastrophic misclassification more heavily
  to improve recall on the highest-stakes tier.
- **Ordinal classification**: exploit tier ordering (0 < 1 < 2 < 3) to penalise
  extreme tier jumps over adjacent-tier errors.
- **Real-time deployment**: FEMA OpenFEMA API provides declaration data within hours —
  a pipeline running this model on each new declaration is technically feasible.